In [50]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns;
import pandas as pd

In [51]:
mel_data = (
    pd.read_excel('Melbourne01.xlsx')
    .rename(columns=lambda col: col.strip())
    .dropna()
)
mel_data.head()

,Year,Month,Day,Hour,Min,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),Wind Gust (km/h),MSLP (hPa),Rainfall since 9 am (mm),gamma,Calculated Dew Pt Temp (degrees C),E (hPa),Calculated Apparent Temp (degrees C)
0,2011,1,1.0,0.0,4.0,24.8,0.0,14.0,51.0,SE,11,13.0,1007.4,0,0.969609,14.1,15.916676,23.9
1,2011,1,1.0,0.0,14.0,24.8,0.0,13.3,48.0,SE,11,11.0,1007.5,0,0.908985,13.2,14.980401,23.6
2,2011,1,1.0,0.0,24.0,24.9,0.0,13.3,48.0,SE,11,13.0,1007.5,0,0.915025,13.2,15.069879,23.7
3,2011,1,1.0,0.0,34.0,24.7,0.0,13.4,49.0,SE,11,11.0,1007.4,0,0.923560,13.4,15.201624,23.6
4,2011,1,1.0,0.0,44.0,24.1,0.0,13.3,51.0,ESE,9,9.0,1007.3,0,0.927209,13.4,15.264860,23.4


In [52]:
numeric_cols = mel_data.columns.drop("Wind Direction")

mel_data[numeric_cols] = mel_data[numeric_cols].apply(
    pd.to_numeric, errors='coerce'
)

mel_data = mel_data.dropna(subset=numeric_cols)

mel_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 504500 entries, 0 to 505354
Data columns (total 18 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Year                                  504500 non-null  int64  
 1   Month                                 504500 non-null  int64  
 2   Day                                   504500 non-null  float64
 3   Hour                                  504500 non-null  float64
 4   Min                                   504500 non-null  float64
 5   Air Temp (degrees C)                  504500 non-null  float64
 6   Apparent Temp (degrees C)             504500 non-null  float64
 7   Dew Pt Temp (degrees C)               504500 non-null  float64
 8   Humidity (%)                          504500 non-null  float64
 9   Wind Direction                        504500 non-null  object 
 10  Wind Speed (km/h)                     504500 non-null  int64  
 11  Wind 

In [53]:
mel_data.rename(columns={'Min': 'Minute'}, inplace=True)

mel_data['Datetime'] = pd.to_datetime(
    mel_data[['Year','Month','Day','Hour','Minute']]
)
mel_data.sort_values('Datetime', inplace=True)
mel_data.Datetime.head()

0   2011-01-01 00:04:00
1   2011-01-01 00:14:00
2   2011-01-01 00:24:00
3   2011-01-01 00:34:00
4   2011-01-01 00:44:00
Name: Datetime, dtype: datetime64[ns]

In [54]:
temp_cols = ['Air Temp (degrees C)', 'Apparent Temp (degrees C)', 'Calculated Apparent Temp (degrees C)']
for col in temp_cols:
    mel_data = mel_data[(mel_data[col] >= -20) & (mel_data[col] <= 55)]

dew_cols = ['Dew Pt Temp (degrees C)', 'Calculated Dew Pt Temp (degrees C)']
for col in dew_cols:
    mel_data = mel_data[mel_data[col] >= -20]
    mel_data = mel_data[mel_data[col] <= mel_data['Air Temp (degrees C)']]

mel_data = mel_data[(mel_data['Humidity (%)'] >= 0) & (mel_data['Humidity (%)'] <= 100)]

mel_data = mel_data[mel_data['Wind Speed (km/h)'] >= 0]
mel_data = mel_data[(mel_data['Wind Gust  (km/h)'] >= 0) & (mel_data['Wind Gust  (km/h)'] <= 120)]

mel_data = mel_data[(mel_data['MSLP (hPa)'] >= 950) & (mel_data['MSLP (hPa)'] <= 1050)]

mel_data = mel_data[(mel_data['E (hPa)'] >= 5) & (mel_data['E (hPa)'] <= 30)]

mel_data = mel_data[(mel_data['gamma'] >= 0.5) & (mel_data['gamma'] <= 2.5)]

mel_data = mel_data[mel_data['Rainfall since 9 am (mm)'] >= 0]

mel_data["Rain"] = (mel_data["Rainfall since 9 am (mm)"] > 0).astype(int)

In [55]:
mel_data['Wind Direction'].unique()

array(['SE', 'ESE', 'NE', 'NNE', 'ENE', 'W', 'N', 'SW', 'SSE', 'S', 'SSW',
       'WSW', 'NW', 'WNW', 'NNW', 'E', 'CALM'], dtype=object)

In [56]:
mel_data

,Year,Month,Day,Hour,Minute,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Direction,Wind Speed (km/h),Wind Gust (km/h),MSLP (hPa),Rainfall since 9 am (mm),gamma,Calculated Dew Pt Temp (degrees C),E (hPa),Calculated Apparent Temp (degrees C),Datetime,Rain
0,2011,1,1.0,0.0,4.0,24.8,0.0,14.0,51.0,SE,11,13.0,1007.4,0.0,0.969609,14.1,15.916676,23.9,2011-01-01 00:04:00,0
1,2011,1,1.0,0.0,14.0,24.8,0.0,13.3,48.0,SE,11,11.0,1007.5,0.0,0.908985,13.2,14.980401,23.6,2011-01-01 00:14:00,0
2,2011,1,1.0,0.0,24.0,24.9,0.0,13.3,48.0,SE,11,13.0,1007.5,0.0,0.915025,13.2,15.069879,23.7,2011-01-01 00:24:00,0
3,2011,1,1.0,0.0,34.0,24.7,0.0,13.4,49.0,SE,11,11.0,1007.4,0.0,0.923560,13.4,15.201624,23.6,2011-01-01 00:34:00,0
4,2011,1,1.0,0.0,44.0,24.1,0.0,13.3,51.0,ESE,9,9.0,1007.3,0.0,0.927209,13.4,15.264860,23.4,2011-01-01 00:44:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505350,2022,3,8.0,15.0,40.0,18.6,16.0,13.2,71.0,SSW,39,46.0,1018.0,0.0,0.917431,13.3,15.179146,12.0,2022-03-08 15:40:00,0
505351,2022,3,8.0,15.0,50.0,18.7,18.5,13.6,72.0,SSW,35,46.0,1017.9,0.0,0.937732,13.6,15.489394,13.0,2022-03-08 15:50:00,0
505352,2022,3,8.0,16.0,0.0,18.9,17.6,13.7,72.0,SSW,35,41.0,1017.8,0.0,0.950348,13.8,15.683896,13.3,2022-03-08 16:00:00,0
505353,2022,3,8.0,16.0,10.0,19.2,18.3,13.8,71.0,S,32,35.0,1017.8,0.0,0.955250,13.9,15.757716,14.2,2022-03-08 16:10:00,0


In [63]:
dummies=pd.get_dummies(mel_data['Wind Direction'], prefix='WindDir')
mel_data = mel_data.join(dummies).drop(columns=['Wind Direction'])

In [64]:
mel_data

,Year,Month,Day,Hour,Minute,Air Temp (degrees C),Apparent Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),...,WindDir_NNW,WindDir_NW,WindDir_S,WindDir_SE,WindDir_SSE,WindDir_SSW,WindDir_SW,WindDir_W,WindDir_WNW,WindDir_WSW
0,2011,1,1.0,0.0,4.0,24.8,0.0,14.0,51.0,11,...,False,False,False,True,False,False,False,False,False,False
1,2011,1,1.0,0.0,14.0,24.8,0.0,13.3,48.0,11,...,False,False,False,True,False,False,False,False,False,False
2,2011,1,1.0,0.0,24.0,24.9,0.0,13.3,48.0,11,...,False,False,False,True,False,False,False,False,False,False
3,2011,1,1.0,0.0,34.0,24.7,0.0,13.4,49.0,11,...,False,False,False,True,False,False,False,False,False,False
4,2011,1,1.0,0.0,44.0,24.1,0.0,13.3,51.0,9,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505350,2022,3,8.0,15.0,40.0,18.6,16.0,13.2,71.0,39,...,False,False,False,False,False,True,False,False,False,False
505351,2022,3,8.0,15.0,50.0,18.7,18.5,13.6,72.0,35,...,False,False,False,False,False,True,False,False,False,False
505352,2022,3,8.0,16.0,0.0,18.9,17.6,13.7,72.0,35,...,False,False,False,False,False,True,False,False,False,False
505353,2022,3,8.0,16.0,10.0,19.2,18.3,13.8,71.0,32,...,False,False,True,False,False,False,False,False,False,False


In [66]:
wind_cols = [c for c in mel_data.columns if c.startswith('WindDir_')]

daily_data = mel_data.resample('D', on='Datetime').agg({
    **{col: 'mean' for col in wind_cols},
    'Air Temp (degrees C)': 'mean',
    'Humidity (%)': 'mean',
    'MSLP (hPa)': 'mean',
    'Wind Speed (km/h)': 'mean',
    'Wind Gust  (km/h)': 'max',
    'E (hPa)': 'mean',
    'Dew Pt Temp (degrees C)': 'mean',
    'Apparent Temp (degrees C)': 'mean',
    'Rainfall since 9 am (mm)': 'sum' 
})
daily_data

,WindDir_CALM,WindDir_E,WindDir_ENE,WindDir_ESE,WindDir_N,WindDir_NE,WindDir_NNE,WindDir_NNW,WindDir_NW,WindDir_S,...,WindDir_WSW,Air Temp (degrees C),Humidity (%),MSLP (hPa),Wind Speed (km/h),Wind Gust (km/h),E (hPa),Dew Pt Temp (degrees C),Apparent Temp (degrees C),Rainfall since 9 am (mm)
Datetime,,,,,,,,,,,,,,,,,,,,,
2011-01-01,0.00,0.000,0.038760,0.015504,0.007752,0.015504,0.007752,0.000000,0.023256,0.418605,...,0.038760,19.651938,61.635659,1011.784496,14.325581,43.0,13.980619,12.004651,0.000000,0.0
2011-01-02,0.00,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.563380,...,0.000000,18.560563,53.718310,1015.983099,22.704225,39.0,11.300188,8.839437,0.000000,0.0
2011-01-03,0.00,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,0.000000,17.300000,51.000000,1016.400000,26.000000,35.0,10.048411,7.100000,0.000000,0.0
2011-01-04,0.00,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008696,0.000000,0.278261,...,0.026087,17.183478,59.608696,1012.045217,15.782609,37.0,11.574564,9.133043,0.000000,0.0
2011-01-05,0.00,0.000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.007752,0.418605,...,0.093023,18.110853,66.565891,1010.086047,19.837209,37.0,13.699222,11.719380,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-03-04,0.12,0.032,0.016000,0.016000,0.272000,0.024000,0.048000,0.064000,0.072000,0.056000,...,0.000000,24.486400,66.512000,1009.955200,15.688000,46.0,19.023386,16.672800,25.424800,20.6
2022-03-05,0.00,0.000,0.015038,0.000000,0.007519,0.022556,0.000000,0.000000,0.007519,0.195489,...,0.210526,19.320301,89.759398,1007.534586,19.406015,43.0,20.054762,17.533835,20.012030,1349.4
2022-03-06,0.00,0.000,0.000000,0.000000,0.007692,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,18.484615,79.061538,1015.901538,27.607692,57.0,16.734805,14.642308,17.083077,188.6


In [68]:
monthly_data = mel_data.resample('ME', on='Datetime').agg({
    **{col: 'mean' for col in wind_cols},
    'Air Temp (degrees C)': 'mean',
    'Humidity (%)': 'mean',
    'MSLP (hPa)': 'mean',
    'Wind Speed (km/h)': 'mean',
    'Wind Gust  (km/h)': 'max',
    'E (hPa)': 'mean',
    'Dew Pt Temp (degrees C)': 'mean',
    'Apparent Temp (degrees C)': 'mean',
    'Rainfall since 9 am (mm)': 'sum'
})
monthly_data

,WindDir_CALM,WindDir_E,WindDir_ENE,WindDir_ESE,WindDir_N,WindDir_NE,WindDir_NNE,WindDir_NNW,WindDir_NW,WindDir_S,...,WindDir_WSW,Air Temp (degrees C),Humidity (%),MSLP (hPa),Wind Speed (km/h),Wind Gust (km/h),E (hPa),Dew Pt Temp (degrees C),Apparent Temp (degrees C),Rainfall since 9 am (mm)
Datetime,,,,,,,,,,,,,,,,,,,,,
2011-01-31,0.000000,0.023851,0.022106,0.009889,0.148051,0.015707,0.045666,0.025015,0.020070,0.206225,...,0.055265,21.447324,64.843805,1011.050553,16.987493,63.0,16.389007,14.041187,0.000000,4297.4
2011-02-28,0.000000,0.017241,0.012106,0.022744,0.169112,0.011372,0.031548,0.041086,0.033382,0.227439,...,0.053925,20.933896,67.639032,1014.094497,15.574101,65.0,16.571908,14.316618,0.000000,1954.2
2011-03-31,0.000000,0.004541,0.010055,0.004217,0.262407,0.014596,0.044437,0.038923,0.019462,0.159260,...,0.092442,19.213039,65.967888,1015.366591,17.373986,72.0,14.488432,12.261953,0.000000,2066.0
2011-04-30,0.000000,0.002705,0.006761,0.005747,0.320487,0.012508,0.048682,0.051386,0.048005,0.082488,...,0.070994,16.358959,68.659567,1022.634990,16.574375,56.0,12.379933,10.124172,0.000000,3866.8
2011-05-31,0.000000,0.001542,0.002057,0.005656,0.183548,0.001028,0.008740,0.024679,0.041645,0.103342,...,0.118766,13.321645,77.299229,1018.192596,14.741902,67.0,11.745212,9.316812,0.000000,2963.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-11-30,0.012996,0.004566,0.006674,0.002107,0.142255,0.027749,0.127151,0.009132,0.014050,0.186512,...,0.041447,16.789463,71.755181,1014.791113,18.714085,82.0,13.284986,10.980295,15.214366,5652.2
2021-12-31,0.017504,0.003388,0.003670,0.011011,0.127612,0.015246,0.072840,0.016375,0.022304,0.250988,...,0.048560,18.740937,63.429983,1014.454433,19.250423,80.0,13.216647,10.999520,17.042377,1994.2
2022-01-31,0.022211,0.012340,0.015548,0.015548,0.104146,0.033564,0.100197,0.017522,0.028134,0.202863,...,0.039733,22.901135,64.495558,1012.425642,17.572310,65.0,17.445545,15.166214,22.946496,6212.4
